# 05 — Unique Analyses
### PM Accelerator — Weather Trend Forecasting Assessment

This notebook implements two unique analyses:
- **A) Feature Importance** — XGBoost feature importances + optional SHAP
- **B) Spatial / Geographical Analysis** — Plotly choropleth world map

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from src.utils import DATA_DIR, FIGURES_DIR, MODELS_DIR, apply_plot_style, save_figure

apply_plot_style()
%matplotlib inline

In [2]:
df = pd.read_csv(
    os.path.join(DATA_DIR, 'cleaned_weather_unscaled.csv'),
    parse_dates=['last_updated'],
)
df.set_index('last_updated', inplace=True)
df.sort_index(inplace=True)
print(f'Shape: {df.shape}')
df.head()

Shape: (130783, 41)


,country,location_name,latitude,longitude,timezone,last_updated_epoch,temperature_celsius,temperature_fahrenheit,condition_text,wind_mph,...,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination,is_outlier
last_updated,,,,,,,,,,,,,,,,,,,,,
2024-05-16 01:45:00,United States of America,Washington Park,46.60,-120.49,America/Los_Angeles,1715849100,16.1,61.0,Clear,4.3,...,7.1,1,1,05:26 AM,08:31 PM,01:36 PM,02:52 AM,Waxing Gibbous,55,False
2024-05-16 02:45:00,El Salvador,San Salvador,13.71,-89.20,America/El_Salvador,1715849100,26.0,78.8,Moderate or heavy rain with thunder,2.2,...,28.1,2,2,05:30 AM,06:16 PM,01:00 PM,01:02 AM,Waxing Gibbous,55,True
2024-05-16 02:45:00,Costa Rica,San Juan,9.97,-84.08,America/Costa_Rica,1715849100,21.0,69.8,Fog,2.2,...,23.3,2,2,05:15 AM,05:51 PM,12:42 PM,12:37 AM,Waxing Gibbous,55,True
2024-05-16 02:45:00,Guatemala,Guatemala City,14.62,-90.53,America/Guatemala,1715849100,20.0,68.0,Mist,13.6,...,178.1,4,10,05:34 AM,06:23 PM,01:05 PM,01:09 AM,Waxing Gibbous,55,True
2024-05-16 02:45:00,Nicaragua,Managua,12.15,-86.27,America/Managua,1715849100,27.2,80.9,Patchy rain nearby,3.6,...,14.7,1,1,05:21 AM,06:02 PM,12:49 PM,12:49 AM,Waxing Gibbous,55,False


---
## Analysis A — Feature Importance (XGBoost)

Train an XGBoost model to predict `temperature_celsius` from all other numeric features, then visualize which features contribute most.

**Note:** We exclude temperature-derived features (`temperature_fahrenheit`, `feels_like_celsius`, `feels_like_fahrenheit`) since they are trivially correlated with the target.

In [3]:
import xgboost as xgb

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
target = 'temperature_celsius'

# Exclude temperature-derived features (leaky), outlier flags, and anomaly labels
exclude_features = [
    target,
    'temperature_fahrenheit',
    'feels_like_celsius',
    'feels_like_fahrenheit',
    'is_outlier',
]
feature_cols = [c for c in numeric_cols 
                if c not in exclude_features and 'anomaly' not in c.lower()]
feature_cols = [c for c in feature_cols if df[c].isnull().mean() < 0.3]

X = df[feature_cols].copy()
y = df[target].copy()
mask = X.isnull().any(axis=1) | y.isnull()
X = X[~mask]
y = y[~mask]

print(f'Features: {len(feature_cols)}')
print(f'Samples:  {len(X)}')
print(f'\nFeatures used: {feature_cols}')

model = xgb.XGBRegressor(
    n_estimators=200, learning_rate=0.05, max_depth=6,
    random_state=42, n_jobs=-1,
)
model.fit(X, y, verbose=False)

importances = pd.Series(model.feature_importances_, index=feature_cols)
importances = importances.sort_values(ascending=True)
top15 = importances.tail(15)
print('\nTop 15 features:')
display(top15.sort_values(ascending=False))

Features: 26
Samples:  130783

Features used: ['latitude', 'longitude', 'last_updated_epoch', 'wind_mph', 'wind_kph', 'wind_degree', 'pressure_mb', 'pressure_in', 'precip_mm', 'precip_in', 'humidity', 'cloud', 'visibility_km', 'visibility_miles', 'uv_index', 'gust_mph', 'gust_kph', 'air_quality_Carbon_Monoxide', 'air_quality_Ozone', 'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10', 'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'moon_illumination']

Top 15 features:


uv_index                        0.319935
pressure_in                     0.222813
latitude                        0.190222
last_updated_epoch              0.046776
humidity                        0.036201
longitude                       0.030823
gust_kph                        0.023212
air_quality_Sulphur_dioxide     0.018840
air_quality_Nitrogen_dioxide    0.016652
cloud                           0.012813
air_quality_Carbon_Monoxide     0.012454
wind_mph                        0.010655
air_quality_PM10                0.009980
visibility_km                   0.009250
air_quality_PM2.5               0.007983
dtype: float32

In [4]:
fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(top15)))
ax.barh(top15.index, top15.values, color=colors, edgecolor='white')
ax.set_title('Top 15 Most Predictive Features for Temperature', fontsize=16, fontweight='bold')
ax.set_xlabel('Feature Importance (Gain)')
ax.grid(True, axis='x', alpha=0.3)
for i, (val, name) in enumerate(zip(top15.values, top15.index)):
    ax.text(val + 0.002, i, f'{val:.3f}', va='center', fontsize=9)
save_figure(fig, 'feature_importance.png')
plt.show()

Figure saved → c:\Users\purva\OneDrive\Desktop\PMA\outputs\figures\feature_importance.png


### Optional: SHAP Analysis

In [5]:
try:
    import shap
    print('Generating SHAP values (this may take a moment)...')
    explainer = shap.TreeExplainer(model)
    X_sample = X.sample(min(1000, len(X)), random_state=42)
    shap_values = explainer.shap_values(X_sample)

    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(shap_values, X_sample, show=False, max_display=15)
    plt.title('SHAP Feature Importance — Beeswarm Plot', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'feature_importance_shap.png'), bbox_inches='tight', dpi=150)
    plt.show()
    print('SHAP plot saved!')
except ImportError:
    print('SHAP not installed — skipping SHAP analysis')
except Exception as e:
    print(f'SHAP analysis failed: {e}')

SHAP not installed — skipping SHAP analysis


In [6]:
pip install plotly

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
pip install pycountry-convert

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ----------------------


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## Analysis B — Spatial / Geographical Analysis (Choropleth)

Interactive world map showing mean temperature by country using Plotly.

In [7]:
import plotly.express as px

country_stats = (
    df.groupby('country')
    .agg(
        mean_temp=('temperature_celsius', 'mean'),
        mean_humidity=('humidity', 'mean'),
        mean_precip=('precip_mm', 'mean'),
        record_count=('temperature_celsius', 'count'),
    )
    .reset_index()
)
print(f'Countries: {len(country_stats)}')

# Get ISO alpha-3 codes
try:
    import pycountry
    def get_iso3(country_name):
        try:
            result = pycountry.countries.search_fuzzy(country_name)
            return result[0].alpha_3
        except Exception:
            return None
    country_stats['iso_alpha'] = country_stats['country'].apply(get_iso3)
    country_stats = country_stats.dropna(subset=['iso_alpha'])
except ImportError:
    print('pycountry not available, using country names directly')
    country_stats['iso_alpha'] = country_stats['country']

Countries: 211


In [12]:
pip install kaleido

Defaulting to user installation because normal site-packages is not writeable

   ---------------------------------------- 0/5 [simplejson]
   ------------------------ --------------- 3/5 [choreographer]
   -------------------------------- ------- 4/5 [kaleido]
   ---------------------------------------- 5/5 [kaleido]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
fig_map = px.choropleth(
    country_stats,
    locations='iso_alpha',
    color='mean_temp',
    hover_name='country',
    hover_data={
        'mean_temp': ':.1f',
        'mean_humidity': ':.1f',
        'mean_precip': ':.2f',
        'record_count': True,
        'iso_alpha': False,
    },
    color_continuous_scale='RdYlBu_r',
    title='Global Mean Temperature by Country',
    labels={'mean_temp': 'Mean Temp (°C)'},
)
fig_map.update_layout(
    geo=dict(showframe=False, showcoastlines=True, projection_type='natural earth'),
    title_font_size=20, width=1200, height=600,
)

# Save as interactive HTML
html_path = os.path.join(FIGURES_DIR, 'spatial_temp_map.html')
fig_map.write_html(html_path)
print(f'Interactive map saved → {html_path}')

# Try saving as PNG too
try:
    png_path = os.path.join(FIGURES_DIR, 'spatial_temp_map.png')
    fig_map.write_image(png_path, width=1200, height=600)
    print(f'Static map saved → {png_path}')
except Exception as e:
    print(f'Could not save static PNG (kaleido needed): {e}')

fig_map.show()
print('\n✅ Unique analyses complete!')

Interactive map saved → c:\Users\purva\OneDrive\Desktop\PMA\outputs\figures\spatial_temp_map.html
Static map saved → c:\Users\purva\OneDrive\Desktop\PMA\outputs\figures\spatial_temp_map.png


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed